# 面试问题：文档 OCR 与 Layout Pipeline 如何恢复阅读序、表格和数值？

可以直接复述的回答是：OCR pipeline 不止识别文字，还要保留 page、bbox、置信度和 region 类型。普通 `y,x` 排序会在双栏页面中左右交错，正确阅读序需先识别跨栏块、栏结构和表格。表格应按行列几何重建，数值纠错必须受字段类型、低置信度与合计约束保护，不能全局把 O 替换成 0。最终答案或下游 chunk 必须带 region provenance。下面用两页、二十个 OCR 区域手写双栏排序、表格重建和低置信度金额修正。

## 真实案例：设备手册双栏页与采购发票表格页

第一页是双栏操作手册，第二页是两行发票表格。区域字段模拟真实 OCR/layout 输出；“1,2O0”故意包含低置信度字母 O。所有公司、金额和编号均为教学构造。

In [1]:
regions = [  # 定义两页二十个带坐标的 OCR 区域
    {"id": "P1-H", "page": 1, "bbox": (0, 0, 100, 10), "type": "title", "text": "设备启动手册", "confidence": 0.99},  # 第一页跨栏标题
    {"id": "P1-L1", "page": 1, "bbox": (5, 15, 45, 23), "type": "heading", "text": "启动步骤", "confidence": 0.98},  # 左栏标题
    {"id": "P1-R1", "page": 1, "bbox": (55, 15, 95, 23), "type": "heading", "text": "安全警告", "confidence": 0.98},  # 右栏标题
    {"id": "P1-L2", "page": 1, "bbox": (5, 26, 45, 34), "type": "paragraph", "text": "一、连接电源", "confidence": 0.97},  # 左栏步骤一
    {"id": "P1-R2", "page": 1, "bbox": (55, 26, 95, 34), "type": "paragraph", "text": "勿在潮湿环境使用", "confidence": 0.96},  # 右栏警告一
    {"id": "P1-L3", "page": 1, "bbox": (5, 38, 45, 46), "type": "paragraph", "text": "二、长按启动键", "confidence": 0.97},  # 左栏步骤二
    {"id": "P1-R3", "page": 1, "bbox": (55, 38, 95, 46), "type": "paragraph", "text": "维护前必须断电", "confidence": 0.96},  # 右栏警告二
    {"id": "P1-F", "page": 1, "bbox": (0, 90, 100, 98), "type": "footer", "text": "第1页", "confidence": 0.99},  # 第一页跨栏页脚
    {"id": "P2-H", "page": 2, "bbox": (0, 0, 100, 10), "type": "title", "text": "采购发票", "confidence": 0.99},  # 第二页标题
    {"id": "P2-C1", "page": 2, "bbox": (5, 15, 45, 23), "type": "table_cell", "row": 0, "col": 0, "text": "项目", "confidence": 0.99},  # 表头项目
    {"id": "P2-C2", "page": 2, "bbox": (45, 15, 65, 23), "type": "table_cell", "row": 0, "col": 1, "text": "数量", "confidence": 0.99},  # 表头数量
    {"id": "P2-C3", "page": 2, "bbox": (65, 15, 95, 23), "type": "table_cell", "row": 0, "col": 2, "text": "金额", "confidence": 0.99},  # 表头金额
    {"id": "P2-A1", "page": 2, "bbox": (5, 27, 45, 35), "type": "table_cell", "row": 1, "col": 0, "text": "传感器", "confidence": 0.97},  # 第一行项目
    {"id": "P2-A2", "page": 2, "bbox": (45, 27, 65, 35), "type": "table_cell", "row": 1, "col": 1, "text": "2", "confidence": 0.98},  # 第一行数量
    {"id": "P2-A3", "page": 2, "bbox": (65, 27, 95, 35), "type": "table_cell", "row": 1, "col": 2, "text": "800", "confidence": 0.97},  # 第一行金额
    {"id": "P2-B1", "page": 2, "bbox": (5, 39, 45, 47), "type": "table_cell", "row": 2, "col": 0, "text": "控制器", "confidence": 0.97},  # 第二行项目
    {"id": "P2-B2", "page": 2, "bbox": (45, 39, 65, 47), "type": "table_cell", "row": 2, "col": 1, "text": "1", "confidence": 0.98},  # 第二行数量
    {"id": "P2-B3", "page": 2, "bbox": (65, 39, 95, 47), "type": "table_cell", "row": 2, "col": 2, "text": "400", "confidence": 0.97},  # 第二行金额
    {"id": "P2-T1", "page": 2, "bbox": (45, 53, 65, 61), "type": "table_cell", "row": 3, "col": 1, "text": "总计", "confidence": 0.98},  # 合计标签
    {"id": "P2-T2", "page": 2, "bbox": (65, 53, 95, 61), "type": "table_cell", "row": 3, "col": 2, "text": "1,2O0", "confidence": 0.72},  # 低置信度 OCR 金额错误
]  # 结束两页区域
print("输入预览：id | page | bbox | type | confidence | text")  # 输出 OCR region 表头
for region in regions:  # 逐条展示二十个版面区域
    print(f"{region['id']} | {region['page']} | {region['bbox']} | {region['type']:10} | {region['confidence']:.2f} | {region['text']}")  # 展示坐标、类型和文字
print("第一页布局：左栏 x<50，右栏 x>=50，标题/页脚跨栏")  # 解释版面坐标语义

输入预览：id | page | bbox | type | confidence | text
P1-H | 1 | (0, 0, 100, 10) | title      | 0.99 | 设备启动手册
P1-L1 | 1 | (5, 15, 45, 23) | heading    | 0.98 | 启动步骤
P1-R1 | 1 | (55, 15, 95, 23) | heading    | 0.98 | 安全警告
P1-L2 | 1 | (5, 26, 45, 34) | paragraph  | 0.97 | 一、连接电源
P1-R2 | 1 | (55, 26, 95, 34) | paragraph  | 0.96 | 勿在潮湿环境使用
P1-L3 | 1 | (5, 38, 45, 46) | paragraph  | 0.97 | 二、长按启动键
P1-R3 | 1 | (55, 38, 95, 46) | paragraph  | 0.96 | 维护前必须断电
P1-F | 1 | (0, 90, 100, 98) | footer     | 0.99 | 第1页
P2-H | 2 | (0, 0, 100, 10) | title      | 0.99 | 采购发票
P2-C1 | 2 | (5, 15, 45, 23) | table_cell | 0.99 | 项目
P2-C2 | 2 | (45, 15, 65, 23) | table_cell | 0.99 | 数量
P2-C3 | 2 | (65, 15, 95, 23) | table_cell | 0.99 | 金额
P2-A1 | 2 | (5, 27, 45, 35) | table_cell | 0.97 | 传感器
P2-A2 | 2 | (45, 27, 65, 35) | table_cell | 0.98 | 2
P2-A3 | 2 | (65, 27, 95, 35) | table_cell | 0.97 | 800
P2-B1 | 2 | (5, 39, 45, 47) | table_cell | 0.97 | 控制器
P2-B2 | 2 | (45, 39, 65, 47) | table_cell | 0.98 | 1
P2-B3 | 2 | 

## Baseline / 基线：按 page、y、x 全局排序并直接拼接

双栏页中相同 y 的左右块会交替出现，启动步骤被安全警告打断；金额字符串也无法解析为整数。

In [2]:
def naive_reading_order(page_regions):  # 实现常见的 y 优先再 x 排序
    return sorted(page_regions, key=lambda region: (region["bbox"][1], region["bbox"][0]))  # 按顶部坐标和左坐标排序
page_one = [region for region in regions if region["page"] == 1]  # 提取双栏手册页区域
naive_page_one = naive_reading_order(page_one)  # 执行忽略栏结构的基线排序
naive_text = " | ".join(region["text"] for region in naive_page_one)  # 拼接基线阅读序文本
total_region = next(region for region in regions if region["id"] == "P2-T2")  # 定位低置信度总计金额区域
try:  # 尝试直接解析 OCR 金额
    naive_total = int(total_region["text"].replace(",", ""))  # 去掉千分位后转整数
except ValueError:  # 捕获字母 O 导致的解析失败
    naive_total = None  # 用空值表示基线无法结构化金额
print("naive page1 order：", [region["id"] for region in naive_page_one])  # 展示左右栏交替顺序
print("naive text：", naive_text)  # 展示被打断的语义段落
print("naive total parse：", naive_total)  # 展示 OCR 字符错误导致数值缺失

naive page1 order： ['P1-H', 'P1-L1', 'P1-R1', 'P1-L2', 'P1-R2', 'P1-L3', 'P1-R3', 'P1-F']
naive text： 设备启动手册 | 启动步骤 | 安全警告 | 一、连接电源 | 勿在潮湿环境使用 | 二、长按启动键 | 维护前必须断电 | 第1页
naive total parse： None


## 核心实现：跨栏块、双栏阅读序与表格矩阵

第一页先输出 title，再完整输出左栏、右栏，最后 footer。第二页 table cell 按显式 row/col 放入矩阵，并保留每格 region ID。

In [3]:
def layout_reading_order(page_regions):  # 按版面结构恢复一个页面阅读序
    titles = sorted([region for region in page_regions if region["type"] == "title"], key=lambda region: region["bbox"][1])  # 提取跨栏标题
    footers = sorted([region for region in page_regions if region["type"] == "footer"], key=lambda region: region["bbox"][1])  # 提取跨栏页脚
    body = [region for region in page_regions if region["type"] not in {"title", "footer"}]  # 提取正文或表格区域
    if page_regions[0]["page"] == 1:  # 对双栏手册页应用列优先排序
        left_column = sorted([region for region in body if region["bbox"][0] < 50], key=lambda region: region["bbox"][1])  # 完整排列左栏
        right_column = sorted([region for region in body if region["bbox"][0] >= 50], key=lambda region: region["bbox"][1])  # 完整排列右栏
        return titles + left_column + right_column + footers  # 返回跨栏标题、左栏、右栏、页脚顺序
    return titles + sorted(body, key=lambda region: (region.get("row", 0), region.get("col", 0))) + footers  # 表格页按行列排序
fixed_page_one = layout_reading_order(page_one)  # 恢复第一页栏级阅读序
table_cells = [region for region in regions if region["page"] == 2 and region["type"] == "table_cell"]  # 提取第二页表格区域
maximum_row = max(region["row"] for region in table_cells)  # 获取表格最大行号
maximum_column = max(region["col"] for region in table_cells)  # 获取表格最大列号
table_matrix = [[None for column in range(maximum_column + 1)] for row in range(maximum_row + 1)]  # 初始化四乘三结构矩阵
provenance_matrix = [[None for column in range(maximum_column + 1)] for row in range(maximum_row + 1)]  # 初始化对应 region ID 矩阵
for region in table_cells:  # 遍历全部 OCR 表格单元
    table_matrix[region["row"]][region["col"]] = region["text"]  # 把文本放入几何行列位置
    provenance_matrix[region["row"]][region["col"]] = region["id"]  # 保存单元格证据身份
print("fixed page1 order：", [region["id"] for region in fixed_page_one])  # 展示完整左栏后右栏顺序
print("fixed text：", " | ".join(region["text"] for region in fixed_page_one))  # 展示连贯手册文本
print("table matrix：")  # 输出重建表格标题
for row, provenance in zip(table_matrix, provenance_matrix):  # 对齐展示每行文本和证据
    print(row, "<-", provenance)  # 展示行列与 region ID

fixed page1 order： ['P1-H', 'P1-L1', 'P1-L2', 'P1-L3', 'P1-R1', 'P1-R2', 'P1-R3', 'P1-F']
fixed text： 设备启动手册 | 启动步骤 | 一、连接电源 | 二、长按启动键 | 安全警告 | 勿在潮湿环境使用 | 维护前必须断电 | 第1页
table matrix：
['项目', '数量', '金额'] <- ['P2-C1', 'P2-C2', 'P2-C3']
['传感器', '2', '800'] <- ['P2-A1', 'P2-A2', 'P2-A3']
['控制器', '1', '400'] <- ['P2-B1', 'P2-B2', 'P2-B3']
[None, '总计', '1,2O0'] <- [None, 'P2-T1', 'P2-T2']


## 低置信度数值修正与六个下游问题

只对 `金额` 列、置信度低于 0.8 且字符形似数字的单元执行 `O→0`，随后用明细金额合计验证。

In [4]:
def parse_money_cell(region):  # 对一个金额单元执行受控 OCR 修正
    raw = region["text"].replace(",", "")  # 去掉千分位分隔符
    corrected = raw  # 默认保持原 OCR 文本
    correction = None  # 初始化无修正事件
    if region.get("col") == 2 and region["confidence"] < 0.80 and all(character.isdigit() or character in {"O", "o"} for character in raw):  # 检查金额列、低置信度和安全字符集
        corrected = raw.replace("O", "0").replace("o", "0")  # 只在门禁内修正常见字母数字混淆
        correction = "O_to_0" if corrected != raw else None  # 记录实际发生的修正
    value = int(corrected) if corrected.isdigit() else None  # 把安全纯数字文本转换为整数
    return value, correction, corrected  # 返回数值、事件和修正字符串
money_regions = [region for region in table_cells if region.get("col") == 2 and region.get("row", 0) > 0]  # 提取明细与合计金额单元
money_ledger = []  # 保存每个金额解析过程
for region in money_regions:  # 遍历三个金额单元
    value, correction, corrected = parse_money_cell(region)  # 执行受控数值解析
    money_ledger.append((region["id"], region["text"], region["confidence"], corrected, value, correction))  # 写入 OCR 修正账本
detail_sum = sum(item[4] for item in money_ledger if item[0] in {"P2-A3", "P2-B3"})  # 计算两条明细金额合计
parsed_total = next(item[4] for item in money_ledger if item[0] == "P2-T2")  # 读取修正后的总计金额
questions = [  # 定义六个下游结构化问题及证据
    ("手册标题", fixed_page_one[0]["text"], [fixed_page_one[0]["id"]]),  # 跨栏标题问答
    ("启动第一步", next(region["text"] for region in fixed_page_one if region["id"] == "P1-L2"), ["P1-L2"]),  # 左栏步骤问答
    ("启动第二步", next(region["text"] for region in fixed_page_one if region["id"] == "P1-L3"), ["P1-L3"]),  # 左栏步骤问答
    ("第一条警告", next(region["text"] for region in fixed_page_one if region["id"] == "P1-R2"), ["P1-R2"]),  # 右栏警告问答
    ("发票项目数", str(2), ["P2-A1", "P2-B1"]),  # 表格行数问答
    ("发票总计", str(parsed_total), ["P2-A3", "P2-B3", "P2-T2"]),  # 带合计校验的金额问答
]  # 结束六个下游问题
print("money ledger：id | raw | confidence | corrected | value | event")  # 输出数值修正账本表头
for item in money_ledger:  # 遍历三个金额单元
    print(item)  # 展示只有低置信度总计被修正
print(f"detail_sum={detail_sum}，parsed_total={parsed_total}，consistent={detail_sum == parsed_total}")  # 展示合计约束验证
print("question | answer | evidence_regions")  # 输出六问题结果表头
for question, answer, evidence in questions:  # 遍历六个结构化输出
    print(f"{question} | {answer} | {evidence}")  # 展示答案与版面证据

money ledger：id | raw | confidence | corrected | value | event
('P2-A3', '800', 0.97, '800', 800, None)
('P2-B3', '400', 0.97, '400', 400, None)
('P2-T2', '1,2O0', 0.72, '1200', 1200, 'O_to_0')
detail_sum=1200，parsed_total=1200，consistent=True
question | answer | evidence_regions
手册标题 | 设备启动手册 | ['P1-H']
启动第一步 | 一、连接电源 | ['P1-L2']
启动第二步 | 二、长按启动键 | ['P1-L3']
第一条警告 | 勿在潮湿环境使用 | ['P1-R2']
发票项目数 | 2 | ['P2-A1', 'P2-B1']
发票总计 | 1200 | ['P2-A3', 'P2-B3', 'P2-T2']


## 失败案例与修正：双栏 y-x 排序造成阅读序交错

正确语义要求“启动步骤→步骤一→步骤二”连续，然后才是安全警告。基线在相同 y 上先左后右，使两个主题交替；栏检测后恢复列优先顺序。

In [5]:
expected_prefix = ["P1-H", "P1-L1", "P1-L2", "P1-L3", "P1-R1", "P1-R2", "P1-R3"]  # 定义双栏正文期望前缀
naive_ids = [region["id"] for region in naive_page_one]  # 提取基线阅读序身份
fixed_ids = [region["id"] for region in fixed_page_one]  # 提取版面修正阅读序身份
naive_left_contiguous = naive_ids[1:4] == ["P1-L1", "P1-L2", "P1-L3"]  # 检查基线左栏是否连续
fixed_left_contiguous = fixed_ids[1:4] == ["P1-L1", "P1-L2", "P1-L3"]  # 检查修正后左栏是否连续
unsafe_global_replacement = "设备型号O型".replace("O", "0")  # 复现全局 O 到零替换破坏普通文本
safe_non_money_region = {"text": "设备型号O型", "confidence": 0.60, "col": 0}  # 构造非金额低置信度文本
safe_value, safe_event, safe_text = parse_money_cell(safe_non_money_region)  # 受控解析拒绝修改非金额字段
print("naive ids：", naive_ids, "左栏连续：", naive_left_contiguous)  # 展示交错失败
print("fixed ids：", fixed_ids, "左栏连续：", fixed_left_contiguous)  # 展示栏级排序修正
print("危险全局替换：", unsafe_global_replacement, "受控非金额结果：", safe_text, safe_event)  # 展示数值修正的字段门禁

naive ids： ['P1-H', 'P1-L1', 'P1-R1', 'P1-L2', 'P1-R2', 'P1-L3', 'P1-R3', 'P1-F'] 左栏连续： False
fixed ids： ['P1-H', 'P1-L1', 'P1-L2', 'P1-L3', 'P1-R1', 'P1-R2', 'P1-R3', 'P1-F'] 左栏连续： True
危险全局替换： 设备型号0型 受控非金额结果： 设备型号O型 None


## 结果解读

OCR 字符只是区域内容，正确文档语义还依赖 bbox、栏和表格行列。版面排序恢复了手册主题连续性，表格矩阵保留每格 provenance。金额修正同时满足列类型、低置信度和安全字符集，并用明细合计二次验证。

## 生产边界

教学区域已经给出 bbox 与类型，真实系统还需检测、旋转校正、文字识别、表格结构、跨页标题和多语言。阅读序可能包含浮动图注和嵌套表格，应使用标注集评估 region recall、CER/WER、reading-order edit distance 与字段准确率。高风险金额需人工复核，不能依靠通用字符替换。

## 最小回归测试

In [6]:
assert len(regions) >= 6 and len(questions) >= 6  # 保证案例包含多个 OCR 区域和下游问题
assert naive_total is None and parsed_total == 1200  # 保证低置信度金额解析失败与修正真实发生
assert detail_sum == parsed_total  # 保证修正总计通过明细合计约束
assert not naive_left_contiguous and fixed_left_contiguous  # 保证双栏阅读序失败和修正可复现
assert fixed_ids[:7] == expected_prefix  # 保证标题、左栏、右栏顺序符合人工版面语义
assert unsafe_global_replacement != "设备型号O型" and safe_text == "设备型号O型"  # 保证字段门禁避免破坏普通文本
assert all(evidence for question, answer, evidence in questions)  # 保证每个下游答案都带 region provenance